# 3장. 데이터의 첫인상 읽기

이 노트북은 강의안 `book/chapters/ch03_data_first_impression.md`를 따라가며 직접 실행해 보는 실습용 자료입니다.

이번 장의 목표는 멋진 분석 결과를 바로 만드는 것이 아니라, 분석 전에 데이터가 어떤 모양인지 차분히 확인하는 습관을 만드는 것입니다. 코드를 한 셀씩 실행하면서 출력 결과를 보고, 바로 아래 설명과 질문에 답해 보세요.


## 0. 이번 장에서 확인할 것

데이터를 처음 열었을 때는 다음 질문에 답할 수 있어야 합니다.

- 데이터 파일은 몇 개인가?
- 각 파일은 어떤 역할을 하는가?
- 각 파일은 몇 행, 몇 열로 구성되어 있는가?
- 어떤 컬럼이 있고, 각 컬럼은 어떤 의미를 가지는가?
- 숫자, 문자, 날짜 컬럼은 무엇인가?
- 비어 있는 값이나 중복된 값은 없는가?
- 여러 파일을 연결할 수 있는 기준 컬럼은 무엇인가?
- LLM이 설명한 데이터 구조가 실제 데이터와 일치하는가?


![CSV 파일을 pandas DataFrame으로 불러오는 흐름](../book/assets/images/ch03/ch03_csv_to_dataframe_flow.svg)

CSV 파일은 텍스트 파일이지만, pandas로 불러오면 행과 열을 가진 `DataFrame`으로 다룰 수 있습니다.


## 1. 실습 준비

먼저 필요한 패키지를 불러오고, 프로젝트 폴더와 데이터 폴더를 찾습니다. 노트북을 `notebooks` 폴더에서 실행해도 되고, 프로젝트 루트에서 실행해도 되도록 `find_project_root()` 함수를 사용합니다.


[💡나의 필기💡] 지난 시간 배운 환경준비

In [33]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)


def find_project_root(start: Path) -> Path:
    """현재 위치에서 위로 올라가며 data/raw 폴더가 있는 프로젝트 루트를 찾습니다."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").exists() and (candidate / "book").exists():
            return candidate
    raise FileNotFoundError("프로젝트 루트를 찾지 못했습니다. 노트북을 저장소 안에서 실행해 주세요.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("프로젝트 루트:", PROJECT_ROOT)
print("데이터 폴더:", DATA_DIR)


프로젝트 루트: D:\git\LLM\llm-data-analysis-course
데이터 폴더: D:\git\LLM\llm-data-analysis-course\data\raw


## 2. 데이터 파일이 있는지 확인하기

파일 경로 오류는 초보자가 가장 자주 만나는 오류입니다. 데이터를 불러오기 전에 필요한 CSV 파일이 실제로 있는지 먼저 확인합니다.


[💡나의 필기💡] 오류나면 ```python scripts/generate_sample_data.py```프로젝트 루트 터미널에서 실행하고,  파일 존재 여부를 다시 확인.

In [34]:
expected_files = [
    "customers.csv",
    "products.csv",
    "orders.csv",
    "order_items.csv",
]

file_check = pd.DataFrame({
    "file": expected_files,
    "path": [str(DATA_DIR / filename) for filename in expected_files],
    "exists": [(DATA_DIR / filename).exists() for filename in expected_files],
})

file_check


,file,path,exists
0,customers.csv,D:\git\LLM\llm-data-analysis-course\data\raw\c...,True
1,products.csv,D:\git\LLM\llm-data-analysis-course\data\raw\p...,True
2,orders.csv,D:\git\LLM\llm-data-analysis-course\data\raw\o...,True
3,order_items.csv,D:\git\LLM\llm-data-analysis-course\data\raw\o...,True


`exists`가 모두 `True`이면 다음 단계로 진행할 수 있습니다.

하나라도 `False`라면 데이터가 아직 생성되지 않았을 수 있습니다. 그 경우 터미널에서 아래 명령을 실행해 샘플 데이터를 생성합니다.

```bash
python scripts/generate_sample_data.py
```


## 3. CSV 파일을 DataFrame으로 불러오기

이제 4개의 CSV 파일을 pandas `DataFrame`으로 불러옵니다. 각 변수 이름은 파일 이름과 비슷하게 맞춰 두면 이후 코드를 읽기 쉽습니다.


In [35]:
customers = pd.read_csv(DATA_DIR / "customers.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
orders = pd.read_csv(DATA_DIR / "orders.csv")
order_items = pd.read_csv(DATA_DIR / "order_items.csv")

print("customers:", type(customers))
print("products:", type(products))
print("orders:", type(orders))
print("order_items:", type(order_items))


customers: <class 'pandas.DataFrame'>
products: <class 'pandas.DataFrame'>
orders: <class 'pandas.DataFrame'>
order_items: <class 'pandas.DataFrame'>


분석할 데이터셋이 여러 개일 때는 딕셔너리로 묶어 두면 반복 점검을 하기 편합니다.


[💡나의 필기💡] 4개의 DataFrame을 datasets에 묶어둠! 
FileNotFoundError이면 STEP 2·3의 경로와 파일 존재 여부를 다시 확인할것!

In [36]:
datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
}

list(datasets.keys())


['customers', 'products', 'orders', 'order_items']

## 4. 각 파일의 역할 이해하기

이번 과정에서 사용하는 데이터는 가상의 온라인 쇼핑몰 운영 데이터입니다.

| 파일 | 역할 | 먼저 확인할 것 |
| --- | --- | --- |
| `customers.csv` | 고객 정보 | 고객 수, 연령, 성별, 지역, 가입일 |
| `products.csv` | 상품 정보 | 상품 수, 카테고리, 가격 |
| `orders.csv` | 주문 정보 | 주문 수, 주문일, 결제수단, 주문상태 |
| `order_items.csv` | 주문 상세 정보 | 주문별 상품, 수량, 단가 |

처음에는 파일을 합치지 말고, 각 파일을 따로 살펴보는 것이 좋습니다.


![pandas DataFrame 구조 예시](../book/assets/images/ch03/ch03_dataframe_structure.svg)

DataFrame은 행(row)과 열(column)로 구성됩니다. `shape`, `head()`, `columns`, `info()` 같은 기본 도구를 사용해 구조를 확인합니다.


## 5. 데이터 크기 확인하기

`shape`는 데이터의 행과 열 개수를 알려 줍니다.

- 앞 숫자: 행 개수
- 뒤 숫자: 열 개수

예를 들어 `(150, 6)`은 150행 6열이라는 뜻입니다.


In [37]:
print("customers:", customers.shape)
print("products:", products.shape)
print("orders:", orders.shape)
print("order_items:", order_items.shape)


customers: (150, 6)
products: (100, 4)
orders: (300, 5)
order_items: (764, 5)


### [shape_summary]

In [38]:
shape_summary = pd.DataFrame([
    {
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
    }
    for name, df in datasets.items()
])

shape_summary


,dataset,rows,columns
0,customers,150,6
1,products,100,4
2,orders,300,5
3,order_items,764,5


### 생각해 보기

- 가장 행이 많은 데이터셋은 무엇인가요?
- `order_items`가 `orders`보다 행이 많다면, 그 이유는 무엇일까요?
- 분석 보고서에 데이터 규모를 설명한다면 어떤 문장으로 쓸 수 있을까요?


#### [💡답변💡]


1. 가장행이 많은 데이터셋:  order_items
2. 주문 1건에 여러 아이템을 주문했을 경우. 
3. 150행 6열  의 `customers`, 100행 4열의 `products`, 300행 5열의 `orders`, 764행 5열의 `order_items`로 구성된 4개의 데이터셋을 사용하였다.
이 중 `order_items`의 행 수가 `orders`보다 많데, 이는 주문 1건에 여러 아이템을 주문했을 경우에 해당한다.
따ㅏ라서, `orders`의 주문 1건과 그 주문에 연결된 `order_items` 여러 건이 매칭되는 관계이다

## 6. 데이터 앞부분과 마지막 부분 보기

`head()`는 앞부분 5행을 보여 줍니다. 컬럼명이 예상과 맞는지, 값의 형태가 자연스러운지 빠르게 확인할 때 사용합니다.


In [39]:
customers.head()


,customer_id,name,gender,age,city,signup_date
0,1,김수민,F,19,광주,2024-08-26
1,2,김정호,F,32,대구,2026-01-09
2,3,이경수,F,61,성남,2024-08-19
3,4,조영호,F,55,울산,2026-06-20
4,5,이예원,F,19,부산,2024-11-20


In [40]:
products.head()


,product_id,product_name,category,price
0,1,전자기기 상품 001,전자기기,160000
1,2,도서 상품 002,도서,34000
2,3,전자기기 상품 003,전자기기,152000
3,4,생활용품 상품 004,생활용품,70000
4,5,식품 상품 005,식품,186000


In [41]:
orders.head()


,order_id,customer_id,order_date,payment_method,order_status
0,1,123,2026-07-14,card,completed
1,2,77,2025-09-29,naver_pay,cancelled
2,3,138,2026-01-26,bank_transfer,cancelled
3,4,57,2026-04-08,kakao_pay,cancelled
4,5,125,2026-02-27,card,cancelled


In [42]:
order_items.head()


,order_item_id,order_id,product_id,quantity,unit_price
0,1,1,100,3,102000
1,2,1,87,5,25000
2,3,1,7,3,142000
3,4,1,9,3,193000
4,5,2,72,4,189000


앞부분만 보고 전체 데이터가 정상이라고 판단하기는 어렵습니다. `tail()`로 마지막 부분도 확인해 봅니다.


In [43]:
customers.tail()


,customer_id,name,gender,age,city,signup_date
145,146,김숙자,M,61,성남,2026-03-01
146,147,이정남,M,19,부산,2025-04-20
147,148,오도현,M,29,고양,2026-08-22
148,149,김정자,M,20,부산,2024-12-26
149,150,조미영,M,40,대전,2026-02-10


### 생각해 보기

`head()`와 `tail()`을 보면서 아래를 확인해 보세요.

- 날짜처럼 보이는 컬럼이 있나요?
- 숫자처럼 보이는 컬럼이 있나요?
- ID처럼 보이는 컬럼이 있나요?
- 사람이 직접 읽을 수 있는 이름이나 범주 값이 있나요?


#### [💡답변💡]

- 날짜처럼 보이는 컬럼:  <br>
       보이는것으로 추정하자면, <br>
              - `orders`에서 `order_date`
              - `customers` 에서 `signup_date` 에 해당하는 열


- 숫자처럼 보이는 컬럼 :  <br>
       보이는것으로 추정하자면,    <br>
              - `customers`의 `customer_id`,`gender`,`age` <br>
              - `products`의 `product_id`,`price` <br>
              - `orders`의 `order_id`,`customer_id` <br>
              - `order_items`의`order_item_id`,`order_id`,`product_id`,`quantity`,`unit_price` <br>


- ID처럼 보이는 컬럼:  <br>
       의미로 추정하자면 <br>
              - `customers`의 `customer_id` <br>
              - `products`의 `product_id` <br>
              - `orders`의 `order_id`,`customer_id` <br>
              - `order_items`의 `order_item_id`, `order_id`, `product_id` <br>


- 사람이 직접 읽을 수 있는 이름이나 범주 값이 있나요? <br>
        모두 읽을 수 있음

## 7. 컬럼명 확인하기

컬럼명은 코드 작성에서 매우 중요합니다. 실제 컬럼명이 `customer_id`인데 LLM이나 사람이 `cust_id`라고 쓰면 코드는 실행되지 않습니다.


In [44]:
for name, df in datasets.items():
    print(f"[{name}]")
    print(list(df.columns))
    print()


[customers]
['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
['product_id', 'product_name', 'category', 'price']

[orders]
['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

[order_items]
['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']



In [45]:
#필기하려고 만든 주석용 복사코드
for name, df in datasets.items():       # datasets 딕셔너리에서 데이터셋 이름(name)과 데이터프레임(df)을 하나씩 꺼냄
    print(f"[{name}]")                  # 현재 확인 중인 데이터셋의 이름을 출력함
    print(list(df.columns))             # 현재 데이터프레임의 컬럼 이름들을 리스트 형태로 출력함
    print()                             # 구분을 위해 빈 줄 출력

[customers]
['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
['product_id', 'product_name', 'category', 'price']

[orders]
['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

[order_items]
['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']



### [column_summary]

In [46]:
column_summary = pd.DataFrame([
    {
        "dataset": name,
        "column_count": len(df.columns),
        "column_names": ", ".join(df.columns),
    }
    for name, df in datasets.items()
])

column_summary


,dataset,column_count,column_names
0,customers,6,"customer_id, name, gender, age, city, signup_date"
1,products,4,"product_id, product_name, category, price"
2,orders,5,"order_id, customer_id, order_date, payment_met..."
3,order_items,5,"order_item_id, order_id, product_id, quantity,..."


### 생각해 보기

- 고객을 구분하는 컬럼은 무엇인가요?
- 주문을 구분하는 컬럼은 무엇인가요?
- 상품을 구분하는 컬럼은 무엇인가요?
- 여러 파일을 연결할 때 사용할 수 있을 것 같은 컬럼은 무엇인가요?


#### [💡답변💡]

- 고객을 구분하는 컬럼: `customer_id`
- 주문을 구분하는 컬럼: `order_id`
- 상품을 구분하는 컬럼: `product_id`
- 여러 파일을 연결할 때 사용할 컬럼: `customer_id`, `order_id`, `product_id` <br>
    `customers` <-> `orders` : `customer_id` 가 공통적으로 존재  <br> 
    `orders` <-> `order_items` : `order_id`가 공통적으로 존재 <br>
    `products` <-> `order_items` : `product_id`가 공통적으로 존재 <br>

## 8. 데이터 타입 확인하기

`info()`는 컬럼별 데이터 타입과 비어 있지 않은 값의 개수를 보여 줍니다.

특히 날짜처럼 보이지만 `object`로 저장된 컬럼을 주의해서 봅니다. pandas에서 `object`는 보통 문자열 또는 여러 타입이 섞인 컬럼일 때 나타납니다.


[💡나의 필기💡] LLM 코드를 고치는 것이 우선!!! 절대 실제 데이터를 LLM 답변에 맞추려고 하지 말것.!!!!!!

In [47]:
customers.info()


<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  150 non-null    int64
 1   name         150 non-null    str  
 2   gender       150 non-null    str  
 3   age          150 non-null    int64
 4   city         150 non-null    str  
 5   signup_date  150 non-null    str  
dtypes: int64(2), str(4)
memory usage: 7.2 KB


In [48]:
for name, df in datasets.items():
    print(f"\n===== {name} =====")
    df.info()



===== customers =====
<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  150 non-null    int64
 1   name         150 non-null    str  
 2   gender       150 non-null    str  
 3   age          150 non-null    int64
 4   city         150 non-null    str  
 5   signup_date  150 non-null    str  
dtypes: int64(2), str(4)
memory usage: 7.2 KB

===== products =====
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   product_id    100 non-null    int64
 1   product_name  100 non-null    str  
 2   category      100 non-null    str  
 3   price         100 non-null    int64
dtypes: int64(2), str(2)
memory usage: 3.2 KB

===== orders =====
<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 5 columns):

In [49]:
dtype_summary = pd.concat(
    [df.dtypes.rename(name) for name, df in datasets.items()],
    axis=1,
).fillna("")

dtype_summary


,customers,products,orders,order_items
customer_id,int64,,int64,
name,str,,,
gender,str,,,
age,int64,,,
city,str,,,
signup_date,str,,,
product_id,,int64,,int64
product_name,,str,,
category,,str,,
price,,int64,,


### 생각해 보기

- 숫자형 컬럼은 어떤 것들이 있나요?
- 문자형 컬럼은 어떤 것들이 있나요?
- 날짜처럼 보이지만 아직 문자열일 가능성이 있는 컬럼은 무엇인가요?


#### [💡답변💡]

- 숫자형 컬럼 (Dtype: int64) <br>
    customers: `customer_id`, `age` <br>
    products: `product_id`, `price` <br>
    orders: `order_id`, `customer_id` <br>
    order_items: `order_item_id`, `order_id`, `product_id`, `quantity`, `unit_price` <br>


- 문자형 컬럼 (Dtype: str) <br>
    customers: `name`, `gender`, `city`, `signup_date` <br>
    products: `product_name`, `category` <br>
    orders: `order_date`, `payment_method`, `order_status` <br>

- 날짜처럼 보이지만 아직 문자열인 컬럼<br>
    `signup_date` <br>
    `order_date` <br>

![데이터 구조 점검 흐름도](../book/assets/images/ch03/ch03_data_check_flow.svg)

데이터 구조 점검은 파일 확인, 로드, 크기 확인, 컬럼 확인, 타입 확인, 결측치 확인, 중복 확인, 키 관계 확인 순서로 진행하면 좋습니다.


## 9. 결측치 확인하기

결측치는 값이 비어 있는 상태입니다. 결측치가 있으면 평균, 비율, 그룹별 집계 결과가 달라질 수 있습니다.

`isna().sum()`은 컬럼별 결측치 개수를 계산합니다.


In [50]:
customers.isna().sum()


customer_id    0
name           0
gender         0
age            0
city           0
signup_date    0
dtype: int64

##### 결측치 개수

In [65]:
missing_summary = pd.concat(
    [df.isna().sum().rename(name) for name, df in datasets.items()],
    axis=1,
).fillna("").astype(str)

missing_summary


#주석 달기위한 복사코드


#결측치 개수
missing_summary = pd.concat(   # 여러 데이터셋의 결측치 개수 결과를 하나의 표로 합침
    [df.isna().sum().rename(name)   # 각 데이터프레임에서 컬럼별 결측치 개수를 계산하고 데이터셋 이름을 붙임
     for name, df in datasets.items()],   # datasets 안의 데이터셋 이름(name)과 데이터프레임(df)을 하나씩 반복해서 처리함
    axis=1,   # 각 데이터셋의 결과를 열 방향으로 나란히 붙임
).fillna("").astype(str)   # 해당 데이터셋에 없는 컬럼에서 생긴 NaN은 빈 문자열로 바꾸고, 전체 값을 문자열로 변환함

missing_summary   # 완성된 결측치 개수 요약표를 출력함



,customers,products,orders,order_items
customer_id,0.0,,0.0,
name,0.0,,,
gender,0.0,,,
age,0.0,,,
city,0.0,,,
signup_date,0.0,,,
product_id,,0.0,,0.0
product_name,,0.0,,
category,,0.0,,
price,,0.0,,


##### 결측치 비율(%)

In [63]:
missing_rate_summary = pd.concat(
    [(df.isna().mean() * 100).round(2).rename(name) for name, df in datasets.items()],
    axis=1,
).fillna("")

missing_rate_summary


#주석 달기위한 복사코드

#결측치 비율(%)
missing_rate_summary = pd.concat(   # 여러 데이터셋의 결측치 비율 결과를 하나의 표로 합침
    [(df.isna().mean() * 100).round(2).rename(name)   # 각 데이터프레임에서 컬럼별 결측치 비율(%)을 계산하고 데이터셋 이름을 붙임
     for name, df in datasets.items()],               # datasets 안의 데이터셋 이름(name)과 데이터프레임(df)을 하나씩 반복해서 처리함
    axis=1,                                           # 계산된 결과들을 열 방향으로 나란히 붙임
).fillna("")                                          # 해당 데이터셋에 없는 컬럼에서 생기는 NaN 값을 빈 문자열로 바꿈

missing_rate_summary                                  # 완성된 결측치 비율 요약표를 출력함




,customers,products,orders,order_items
customer_id,0.0,,0.0,
name,0.0,,,
gender,0.0,,,
age,0.0,,,
city,0.0,,,
signup_date,0.0,,,
product_id,,0.0,,0.0
product_name,,0.0,,
category,,0.0,,
price,,0.0,,


### 결측치 해석 팁

결측치가 있다고 해서 무조건 삭제하는 것은 아닙니다.

| 처리 방법 | 설명 |
| --- | --- |
| 행 제외 | 결측치가 있는 행을 분석에서 제외합니다. |
| 대표값 대체 | 평균, 중앙값, 최빈값 등으로 채웁니다. |
| 별도 범주 처리 | `Unknown` 같은 범주로 표시합니다. |
| 컬럼 제외 | 분석 목적에 맞지 않는 컬럼은 사용하지 않습니다. |
| 원인 확인 | 수집 과정에서 문제가 있었는지 확인합니다. |


##### [💡나의 노트💡]

```py
isna()       → 결측치인지 확인
sum()        → 결측치 개수 계산
rename()     → 데이터셋 이름 붙이기
concat()     → 결과 합치기
fillna("")   → 불필요한 NaN을 빈칸으로
astype(str)  → 문자열로 변환
```

## 10. 중복 데이터 확인하기

중복은 같은 행이나 같은 ID가 반복되는 상태입니다.

단, 모든 중복이 오류는 아닙니다. 예를 들어 `order_items`에서는 한 주문에 여러 상품이 들어갈 수 있으므로 같은 `order_id`가 여러 번 나올 수 있습니다.


In [62]:
duplicate_rows = pd.DataFrame([
    {
        "dataset": name,
        "duplicated_rows": df.duplicated().sum(),
    }
    for name, df in datasets.items()
])

duplicate_rows


#주석 달기위한 복사코드

duplicate_rows = pd.DataFrame([   # 각 데이터셋의 중복 행 개수를 정리해서 새로운 데이터프레임으로 만듦
    {   # 데이터셋 하나에 대한 결과를 딕셔너리 형태로 저장
        "dataset": name,   # 현재 데이터셋의 이름을 dataset 컬럼에 저장
        "duplicated_rows": df.duplicated().sum(),   # 완전히 동일한 중복 행의 개수를 계산해서 저장
    }
    for name, df in datasets.items()   # datasets 안의 모든 데이터셋을 하나씩 반복해서 처리
])

duplicate_rows   # 요약표 출력

,dataset,duplicated_rows
0,customers,0
1,products,0
2,orders,0
3,order_items,0


In [61]:
id_duplicate_checks = pd.DataFrame([
    {
        "check": "customers.customer_id",
        "duplicated_count": customers["customer_id"].duplicated().sum(),
        "interpretation": "0이어야 고객 ID가 유일합니다.",
    },
    {
        "check": "products.product_id",
        "duplicated_count": products["product_id"].duplicated().sum(),
        "interpretation": "0이어야 상품 ID가 유일합니다.",
    },
    {
        "check": "orders.order_id",
        "duplicated_count": orders["order_id"].duplicated().sum(),
        "interpretation": "0이어야 주문 ID가 유일합니다.",
    },
    {
        "check": "order_items.order_id",
        "duplicated_count": order_items["order_id"].duplicated().sum(),
        "interpretation": "한 주문에 여러 상품이 있으면 0보다 클 수 있습니다.",
    },
])

id_duplicate_checks



#주석 달기위한 복사코드
id_duplicate_checks = pd.DataFrame([   # ID 컬럼별 중복 개수를 정리해서 새로운 데이터프레임으로 만듦
    {
        "check": "customers.customer_id",   # customers 데이터의 customer_id 컬럼을 검사한다는 의미
        "duplicated_count": customers["customer_id"].duplicated().sum(),   # customer_id에서 중복된 값의 개수를 계산함
        "interpretation": "0이어야 고객 ID가 유일합니다.",   # 중복 개수가 0이면 모든 고객 ID가 서로 다르다는 의미
    },

    {
        "check": "products.product_id",   # products 데이터의 product_id 컬럼을 검사
        "duplicated_count": products["product_id"].duplicated().sum(),   # product_id에서 중복된 값의 개수를 계산함
        "interpretation": "0이어야 상품 ID가 유일합니다.",   # 중복 개수가 0이면 모든 상품 ID가 유일함
    },

    {
        "check": "orders.order_id",   # orders 데이터의 order_id 컬럼을 검사
        "duplicated_count": orders["order_id"].duplicated().sum(),    # order_id에서 중복된 값의 개수를 계산함
        "interpretation": "0이어야 주문 ID가 유일합니다.",   # 중복 개수가 0이면 각각의 주문이 고유한 order_id를 가짐
    },

    {
        "check": "order_items.order_id",   # order_items 데이터의 order_id 컬럼을 검사
        "duplicated_count": order_items["order_id"].duplicated().sum(),      # order_items에서 같은 order_id가 반복되는 개수를 계산함
        "interpretation": "한 주문에 여러 상품이 있으면 0보다 클 수 있습니다.",    # 하나의 주문에 여러 상품이 포함될 수 있으므로 order_id 중복은 정상일 수 있음
    },

])

id_duplicate_checks   # ID 중복 검사 결과를 표 형태로 출력


,check,duplicated_count,interpretation
0,customers.customer_id,0,0이어야 고객 ID가 유일합니다.
1,products.product_id,0,0이어야 상품 ID가 유일합니다.
2,orders.order_id,0,0이어야 주문 ID가 유일합니다.
3,order_items.order_id,464,한 주문에 여러 상품이 있으면 0보다 클 수 있습니다.


In [162]:
# 전체 행 중복 확인
for name, df in datasets.items():   # datasets에 있는 각 데이터셋을 하나씩 반복
    print(name, df.duplicated().sum())   # 데이터셋 이름과 완전히 동일한 중복 행의 개수를 출력


# 주요 ID 컬럼 지정
key_columns = {
    "customers": "customer_id",        # customers의 주요 ID는 customer_id
    "products": "product_id",          # products의 주요 ID는 product_id
    "orders": "order_id",              # orders의 주요 ID는 order_id
    "order_items": "order_item_id",    # order_items의 주요 ID는 order_item_id
}

print(f"\n===== 주요 ID =====")

# 주요 ID의 결측치와 중복 여부 확인
for name, key in key_columns.items():   # 데이터셋 이름(name)과 주요 ID 컬럼(key)을 하나씩 반복
    df = datasets[name]                 # 해당 이름의 데이터프레임을 datasets에서 가져옴
    print(
        name,                                       # 현재 데이터셋 이름 출력
        "missing:", df[key].isna().sum(),", "           # 주요 ID 컬럼에서 결측치가 몇 개인지 계산
        "duplicated:", df[key].duplicated().sum(),  # 주요 ID 컬럼에서 중복된 값이 몇 개인지 계산
    )

customers 0
products 0
orders 0
order_items 0

===== 주요 ID =====
customers missing: 0 , duplicated: 0
products missing: 0 , duplicated: 0
orders missing: 0 , duplicated: 0
order_items missing: 0 , duplicated: 0


### 생각해 보기

- `customers.customer_id` 중복과 `order_items.order_id` 중복은 왜 의미가 다를까요?
- 중복 개수만 보고 삭제하면 위험한 이유는 무엇일까요?


#### [💡답변💡]

 정상적인 반복값까지 잘못 삭제할 수 있기 때문에 중복 개수만 보고 바로 삭제하면 위험할 수 있다. <br>
 `customers.customer_id` 는 고객을 고유하게 식별하는 컬럼이므로 중복이 발생하면 데이터 이상 여부를 확인할 필요가 있다.  <br>
 반면 `order_items.order_id` 는 하나의 주문에 포함된 여러 상품 행에서 반복될 수 있으므로 중복이 정상적으로 발생할 수 있다.  <br>
 따라서 중복 개수만을 기준으로 데이터를 삭제해서는 안 되며 해당 컬럼의 역할과 테이블 간 관계를 먼저 확인해야 한다.  <br>
 그렇지 않으면 정상적인 주문 상세 데이터까지 삭제하여 분석 결과가 왜곡될 수 있다. <br>

## 11. 숫자형 컬럼 기본 통계 확인하기

`describe()`는 숫자형 컬럼의 개수, 평균, 표준편차, 최솟값, 사분위수, 최댓값을 보여 줍니다.

최솟값이나 최댓값이 지나치게 이상하면 데이터 오류나 이상치 가능성을 의심할 수 있습니다.


In [66]:
customers.describe()


,customer_id,age
count,150.000000,150.000000
mean,75.500000,42.086667
std,43.445368,15.613166
min,1.000000,19.000000
25%,38.250000,29.000000
50%,75.500000,40.000000
75%,112.750000,57.000000
max,150.000000,69.000000


In [67]:
products[["price"]].describe()


,price
count,100.000000
mean,110040.000000
std,56433.910574
min,5000.000000
25%,65750.000000
50%,112000.000000
75%,161000.000000
max,200000.000000


In [68]:
order_items[["quantity", "unit_price"]].describe()


,quantity,unit_price
count,764.000000,764.000000
mean,3.053665,108561.518325
std,1.410873,56996.770604
min,1.000000,5000.000000
25%,2.000000,62000.000000
50%,3.000000,111000.000000
75%,4.000000,161250.000000
max,5.000000,200000.000000


숫자가 문자열로 저장된 경우도 있습니다. 예를 들어 `"10,000"`처럼 쉼표가 포함된 문자열은 바로 계산하기 어렵습니다.


In [70]:
price_text = pd.Series(["10,000", "25,500", "3000", "확인필요"])
price_number = pd.to_numeric(
    price_text.str.replace(",", "", regex=False),
    errors="coerce",
)

pd.DataFrame({
    "original": price_text,
    "converted": price_number,
})


,original,converted
0,"10,000",10000.0
1,"25,500",25500.0
2,3000,3000.0
3,확인필요,NaN


`errors="coerce"`는 숫자로 바꿀 수 없는 값을 `NaN`으로 처리합니다. 변환 후에는 새로 생긴 결측치가 있는지도 확인해야 합니다.


In [ ]:
price_text = pd.Series(["10,000", "25,500", "3000", "확인필요"])# 문자열 형태의 가격 데이터를 Series로 생성
price_number = pd.to_numeric(    # 문자열 데이터를 숫자형으로 변환
    price_text.str.replace(",", "", regex=False),    # 각 문자열에서 쉼표(,)를 제거 (10,000 -> 10000)
    errors="coerce",    # 숫자로 변환할 수 없는 값은 오류를 내지 않고 NaN으로 바꿈 
)

pd.DataFrame({    # 원본과 변환 결과를 비교할 수 있도록 DataFrame 생성
    "original": price_text,    # 변환 전 원본 문자열
    "converted": price_number,    # 숫자로 변환된 결과
})

In [84]:
#숫자형 컬럼에 대한 기초 통계량을 확인

customers[["age"]].describe() # customers 데이터에서 age 컬럼만 선택하고 개수, 평균, 표준편차, 최솟값, 사분위수, 최댓값을 확인
products[["price"]].describe() # products 데이터에서 price 컬럼만 선택한 뒤 가격의 기초 통계량을 확인
order_items[["quantity", "unit_price"]].describe() # order_items 데이터에서 quantity와 unit_price 컬럼만 선택한 뒤 각각의 기초 통계량을 확인

,quantity,unit_price
count,764.000000,764.000000
mean,3.053665,108561.518325
std,1.410873,56996.770604
min,1.000000,5000.000000
25%,2.000000,62000.000000
50%,3.000000,111000.000000
75%,4.000000,161250.000000
max,5.000000,200000.000000


In [89]:
#범주형 컬럼에서 각 값이 몇 번 등장하는지 개수를 셈


#dropna=False => 결측치도 제외하지 말고 함께 세라
# value_counts() => 각 고유값의 등장 횟수
customers["city"].value_counts(dropna=False) # customers의 city 컬럼에서 각 도시가 몇 번 등장하는지 개수를 셈. 
products["category"].value_counts(dropna=False) # products의 category 컬럼에서 각 카테고리가 몇 번 등장하는지 개수를 셈. 
orders["order_status"].value_counts(dropna=False) # orders의 order_status 컬럼에서 각 상태가 몇 번 등장하는지 개수를 셈.

order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

## 12. 범주형 컬럼 고유값 확인하기

문자형 또는 범주형 컬럼은 고유값 개수와 빈도를 확인합니다. 예를 들어 지역, 성별, 카테고리, 주문 상태 같은 컬럼은 `value_counts()`로 분포를 볼 수 있습니다.


In [90]:
customers["city"].value_counts().head(10)


city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

In [91]:
products["category"].value_counts()


category
스포츠     19
전자기기    17
생활용품    16
뷰티      16
도서      14
패션      11
식품       7
Name: count, dtype: int64

In [92]:
orders["order_status"].value_counts()


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

In [93]:
categorical_summary = pd.DataFrame([
    {"dataset": "customers", "column": "city", "unique_count": customers["city"].nunique()},
    {"dataset": "customers", "column": "gender", "unique_count": customers["gender"].nunique()},
    {"dataset": "products", "column": "category", "unique_count": products["category"].nunique()},
    {"dataset": "orders", "column": "payment_method", "unique_count": orders["payment_method"].nunique()},
    {"dataset": "orders", "column": "order_status", "unique_count": orders["order_status"].nunique()},
])

categorical_summary


,dataset,column,unique_count
0,customers,city,10
1,customers,gender,2
2,products,category,7
3,orders,payment_method,4
4,orders,order_status,3


### 생각해 보기

- 특정 값에 데이터가 지나치게 몰려 있나요?
- 오타나 표기 차이처럼 보이는 값이 있나요?
- 나중에 그룹별 분석 기준으로 쓰기 좋은 컬럼은 무엇인가요?


#### [💡답변💡]

- 특정 값에 데이터가 지나치게 몰려 있는가?
`order_status`에서는 completed가 184건으로 전체 300건 중 약 61.3%를 차지하여 다른 상태보다 높은 비중을 보인다. <br>
반면 상품 카테고리는 스포츠 19개, 전자기기 17개, 생활용품과 뷰티가 각각 16개 등으로 비교적 여러 범주에 분산되어 있다. <br>
고객 지역도 성남이 21명으로 가장 많지만 전체 150명 중 14% 수준이므로 특정 지역에 지나치게 집중되어 있다고 보기는 어렵다. <br>

- 오타나 표기 차이처럼 보이는 값이 있는가?
현재 확인한 city, category, order_status의 출력 결과에서는 눈에 띄는 오타나 표기 차이가 보이지 않는다.  <br>
다만 gender와 payment_method는 고유값의 개수만 확인했기 때문에 실제 값의 표기가 일관적인지는 현재 결과만으로 확인할 수 없다.  <br>
`value_counts()` 또는 `unique()`로 실제 값을 추가 확인할 필요가 있다. <br>

- 나중에 그룹별 분석 기준으로 사용하기 좋은 컬럼은 무엇인가?
city, gender, category, payment_method, order_status와 같은 범주형 컬럼을 그룹별 분석 기준으로 활용할 수 있다.  <br>
예를 들어 지역별 고객 수, 성별 구매 특성, 카테고리별 매출, 결제수단별 주문 건수, 주문 상태별 주문 비중 등을 비교할 수 있다. <br>
특히 order_status는 나중에 실제 매출 분석을 한다면 cancelled, refunded 주문을 포함할지 제외할지 분석 목적에 따라 먼저 결정해야 하기때문에 분석시 용이하게 쓰일 것으로 예상된다.

## 13. 날짜 컬럼 확인하기

날짜 컬럼은 월별, 요일별, 기간별 분석에 자주 사용됩니다.

하지만 CSV에서 읽어온 날짜는 처음에는 문자열(`object`)일 수 있습니다. `pd.to_datetime()`으로 날짜 타입으로 바꿔야 날짜 계산을 안전하게 할 수 있습니다.


In [94]:
orders["order_date"].head()


0    2026-07-14
1    2025-09-29
2    2026-01-26
3    2026-04-08
4    2026-02-27
Name: order_date, dtype: str

In [108]:
print("변환 전 타입:", orders["order_date"].dtype)

orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")

print("변환 후 타입:", orders["order_date"].dtype)
print("날짜 변환 실패 건수:", orders["order_date"].isna().sum())
print("가장 빠른 주문일:", orders["order_date"].min())
print("가장 최근 주문일:", orders["order_date"].max())

#======================================================================================
#=================================주석 달기위한 복사코드=================================
#======================================================================================

print("변환 전 타입:", orders["order_date"].dtype)# order_date 컬럼의 현재 자료형을 확인
orders["order_date"] = pd.to_datetime(
    orders["order_date"],     # order_date 컬럼을 날짜형으로 변환
    errors="coerce"           # 날짜로 변환할 수 없는 값은 NaT(날짜형 결측치)로 처리
)
print("변환 후 타입:", orders["order_date"].dtype)                  # 날짜형으로 제대로 변환되었는지 자료형을 다시 확인
print("날짜 변환 실패 건수:", orders["order_date"].isna().sum())     # 변환 후 NaT가 된 값의 개수를 세어 날짜 변환 실패 건수를 확인
print("가장 빠른 주문일:", orders["order_date"].min())               # order_date 중 가장 이른 날짜를 확인
print("가장 최근 주문일:", orders["order_date"].max())               # order_date 중 가장 늦은 날짜를 확인


변환 전 타입: datetime64[us]
변환 후 타입: datetime64[us]
날짜 변환 실패 건수: 0
가장 빠른 주문일: 2025-09-15 00:00:00
가장 최근 주문일: 2026-09-14 00:00:00


In [107]:
orders.assign(
    order_year=orders["order_date"].dt.year,
    order_month=orders["order_date"].dt.month,
    order_day_name=orders["order_date"].dt.day_name(),
).head()


#======================================================================================
#=================================주석 달기위한 복사코드=================================
#======================================================================================
orders.assign(
    order_year=orders["order_date"].dt.year,    # order_date에서 연도만 추출해서 order_year라는 임시 컬럼 생성
    order_month=orders["order_date"].dt.month,    # order_date에서 월만 추출해서 order_month라는 임시 컬럼 생성
    order_day_name=orders["order_date"].dt.day_name(),    # order_date에서 요일 이름을 추출해서 order_day_name이라는 임시 컬럼 생성
).head() # 새 컬럼이 추가된 결과 중 앞의 5행만 확인


,order_id,customer_id,order_date,payment_method,order_status,order_year,order_month,order_day_name
0,1,123,2026-07-14,card,completed,2026,7,Tuesday
1,2,77,2025-09-29,naver_pay,cancelled,2025,9,Monday
2,3,138,2026-01-26,bank_transfer,cancelled,2026,1,Monday
3,4,57,2026-04-08,kakao_pay,cancelled,2026,4,Wednesday
4,5,125,2026-02-27,card,cancelled,2026,2,Friday


### 생각해 보기

- 데이터는 어느 기간을 포함하고 있나요?
- 월별 매출 분석을 하기에 충분한 기간인가요?
- 날짜 변환 실패 건수가 0보다 크다면 무엇을 확인해야 할까요?


#### [💡답변💡]

주문 데이터의 기간은 2025년 9월 15일부터 2026년 9월 14일까지로 확인되었다. <br>
날짜형 데이터를 활용하면 연도, 월, 요일 등의 파생변수를 생성할 수 있어 월별 주문 추이 또는 요일별 주문 패턴과 분석에 활용할 수 있다. <br>
`order_date` 컬럼은 이미 `datetime64[us]` 형식으로 저장되어 있었으며, 날짜 변환 과정에서 실패한 값은 0건이었다.  <br>
만약 날짜 변환 실패 건수가 0보다 크다면, 날짜 형식이 일관되지 않거나 잘못된 날짜, 날짜가 아닌 문자열, 빈 값 등이 포함되어 있는지 확인해야 한다.  <br>
또한 변환에 실패한 원본 값을 직접 확인하여 데이터 입력 오류인지, 별도의 전처리가 필요한 값인지 판단해야 한다. <br>


### 변환 실패값 확인 하기.

In [109]:
#변환 실패값 확인 하기.

converted = pd.to_datetime(orders["order_date"], errors="coerce")
orders.loc[converted.isna(), "order_date"] # 날짜 변환에 실패한 원본 값만 확인

Series([], Name: order_date, dtype: datetime64[us])

#### 가입일도 별도로 확인

In [115]:
#가입일도 별도로 확인
signup_check = pd.to_datetime(customers["signup_date"],errors="coerce",)
print("가입일 변환 실패:", signup_check.isna().sum())

가입일 변환 실패: 0


## 14. 여러 파일의 관계 확인하기

온라인 쇼핑몰 데이터는 고객, 상품, 주문, 주문 상세 데이터가 서로 연결되어야 분석할 수 있습니다.

| 연결 관계 | 의미 |
| --- | --- |
| `customers.customer_id` ↔ `orders.customer_id` | 어떤 고객이 주문했는지 연결합니다. |
| `orders.order_id` ↔ `order_items.order_id` | 주문과 주문 상세를 연결합니다. |
| `products.product_id` ↔ `order_items.product_id` | 주문 상세와 상품 정보를 연결합니다. |


![4개 CSV 파일 간 키 관계도](../book/assets/images/ch03/ch03_csv_key_relationships.svg)


In [110]:
invalid_customers = orders[~orders["customer_id"].isin(customers["customer_id"])]
invalid_orders = order_items[~order_items["order_id"].isin(orders["order_id"])]
invalid_products = order_items[~order_items["product_id"].isin(products["product_id"])]

relationship_check = pd.DataFrame([
    {
        "relationship": "orders.customer_id -> customers.customer_id",
        "invalid_rows": len(invalid_customers),
    },
    {
        "relationship": "order_items.order_id -> orders.order_id",
        "invalid_rows": len(invalid_orders),
    },
    {
        "relationship": "order_items.product_id -> products.product_id",
        "invalid_rows": len(invalid_products),
    },
])

relationship_check


,relationship,invalid_rows
0,orders.customer_id -> customers.customer_id,0
1,order_items.order_id -> orders.order_id,0
2,order_items.product_id -> products.product_id,0


`invalid_rows`가 모두 0이면 샘플 데이터에서는 기본적인 연결 관계가 유지되고 있다고 볼 수 있습니다. 0보다 큰 값이 있다면 어느 파일에서 기준 ID가 빠져 있는지 먼저 확인해야 합니다.


#### [💡나의 필기💡]

```txt
A["A-a"]          → 검사할 값
B["B-a"]          → 비교할 기준 목록
.isin(...)        → 기준 목록에 존재하는지 확인
~                 → 결과를 반대로 뒤집어 '존재하지 않는 값' 선택
A.loc[...]        → 해당 조건을 만족하는 A의 행 추출
```


==============================
```py
# A.loc[~A["A-a"].isin(B["B-a"])] 
## A 데이터프레임의 A-a 값이 B 데이터프레임의 B-a 목록에 존재하지 않는 행만 찾음
# 기준 데이터프레임의 특정 컬럼 값이 참조 데이터프레임의 특정 컬럼에 존재하지 않는 행을 추출

invalid_customers = orders.loc[~orders["customer_id"].isin(customers["customer_id"])]# 실제 고객 테이블에 없는 고객 ID로 생성된 주문이 있는지 확인
invalid_orders = order_items.loc[~order_items["order_id"].isin(orders["order_id"])]# 실제 주문 테이블에 없는 주문 ID가 주문 상세에 있는지 확인
invalid_products = order_items.loc[~order_items["product_id"].isin(products["product_id"])]# 상품 테이블에 없는 상품 ID가 주문 상세에 있는지 확인

print("없는 customer_id:", len(invalid_customers))# 존재하지 않는 customer_id를 가진 주문 행의 개수를 출력
print("없는 order_id:", len(invalid_orders))# 존재하지 않는 order_id를 가진 주문 상세 행의 개수를 출력
print("없는 product_id:", len(invalid_products))# 존재하지 않는 product_id를 가진 주문 상세 행의 개수를 출력

```

In [160]:
# A.loc[~A["A-a"].isin(B["B-a"])] 
## A 데이터프레임의 A-a 값이 B 데이터프레임의 B-a 목록에 존재하지 않는 행만 찾음
# 기준 데이터프레임의 특정 컬럼 값이 참조 데이터프레임의 특정 컬럼에 존재하지 않는 행을 추출

invalid_customers = orders.loc[~orders["customer_id"].isin(customers["customer_id"])]# 실제 고객 테이블에 없는 고객 ID로 생성된 주문이 있는지 확인
invalid_orders = order_items.loc[~order_items["order_id"].isin(orders["order_id"])]# 실제 주문 테이블에 없는 주문 ID가 주문 상세에 있는지 확인
invalid_products = order_items.loc[~order_items["product_id"].isin(products["product_id"])]# 상품 테이블에 없는 상품 ID가 주문 상세에 있는지 확인

print("없는 customer_id:", len(invalid_customers))# 존재하지 않는 customer_id를 가진 주문 행의 개수를 출력
print("없는 order_id:", len(invalid_orders))# 존재하지 않는 order_id를 가진 주문 상세 행의 개수를 출력
print("없는 product_id:", len(invalid_products))# 존재하지 않는 product_id를 가진 주문 상세 행의 개수를 출력

없는 customer_id: 0
없는 order_id: 0
없는 product_id: 0


## 15. 간단한 병합으로 관계 확인하기

키 관계가 맞는지 확인한 뒤에는 데이터를 병합해 볼 수 있습니다. 아래 코드는 주문 상세(`order_items`)에 상품 정보(`products`)를 붙이고, 각 행의 금액을 계산합니다.


In [134]:
order_items_with_products = order_items.merge(
    products,
    on="product_id",
    how="left",
)

order_items_with_products["line_amount"] = (
    order_items_with_products["quantity"] * order_items_with_products["unit_price"]
)

order_items_with_products.head()

#주석 달기위한 복사코드
order_items_with_products = order_items.merge(    # order_items와 products 데이터프레임을 병합
    products,    # order_items에 붙일 대상 데이터프레임
    on="product_id",    # 두 데이터프레임의 product_id 값이 같은 행끼리 연결
    how="left",    # order_items의 모든 행은 유지하는 left join 방식으로 병합
)

order_items_with_products["line_amount"] = (    # line_amount라는 새로운 컬럼을 생성
    order_items_with_products["quantity"]    # 각 주문 상세 행의 상품 수량을 가져옴
    * order_items_with_products["unit_price"]    # 수량에 판매 단가를 곱해서 해당 행의 매출액을 계산
)

order_items_with_products.head()# 상품 정보와 매출액이 추가된 데이터의 앞 5행을 확인

,order_item_id,order_id,product_id,quantity,unit_price,product_name,category,price,line_amount
0,1,1,100,3,102000,도서 상품 100,도서,102000,306000
1,2,1,87,5,25000,도서 상품 087,도서,25000,125000
2,3,1,7,3,142000,도서 상품 007,도서,142000,426000
3,4,1,9,3,193000,스포츠 상품 009,스포츠,193000,579000
4,5,2,72,4,189000,뷰티 상품 072,뷰티,189000,756000


#### [💡나의 노트💡]

```py
A.merge(B, on="공통컬럼", how="left")
```
A 데이터프레임을 기준으로, A와 B의 공통 컬럼 값이 같은 행끼리 연결하여 B의 정보를 A에 추가한다.

In [136]:

#상품 카테고리별 매출 총액을 계산한 뒤 매출이 높은 순서대로 정렬
category_sales = (
    order_items_with_products
    .groupby("category", as_index=False)["line_amount"]
    .sum()
    .sort_values("line_amount", ascending=False)
)

category_sales

#주석 달기위한 복사코드
category_sales = (order_items_with_products    # 상품 정보와 매출액(line_amount)이 포함된 데이터프레임을 사용
    .groupby("category", # category별로 데이터를 그룹화하고, 각 그룹에서 line_amount 컬럼만 선택
              as_index=False)["line_amount"] # as_index=False는 category를 인덱스가 아니라 일반 컬럼으로 유지
    .sum() # 각 category별 line_amount를 모두 더해서 카테고리별 총매출을 계산
    .sort_values("line_amount", ascending=False)  # 총매출(line_amount)이 큰 카테고리부터 내림차순으로 정렬
)

category_sales# 계산된 카테고리별 총매출 결과를 출력


,category,line_amount
3,스포츠,50174000
1,뷰티,47551000
5,전자기기,41003000
2,생활용품,34839000
4,식품,33597000
0,도서,24645000
6,패션,23801000


이 결과는 본격적인 EDA가 아니라, 데이터 관계가 실제로 연결되는지 확인하는 작은 점검입니다. 분석 결론을 내리기 전에 결측치, 주문 상태, 취소 주문 처리 기준 등을 더 확인해야 합니다.


## 16. 반복 점검을 함수로 정리하기

여러 데이터셋에 같은 점검을 반복할 때는 함수로 정리하면 편합니다.


In [138]:
def check_data_overview(name: str, df: pd.DataFrame) -> None:
    print(f"===== {name} =====")
    print("shape:", df.shape)
    print("\ncolumns:")
    print(list(df.columns))
    print("\ndtypes:")
    print(df.dtypes)
    print("\nmissing values:")
    print(df.isna().sum())
    print("\nduplicated rows:", df.duplicated().sum())


check_data_overview("customers", customers)


===== customers =====
shape: (150, 6)

columns:
['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

dtypes:
customer_id    int64
name             str
gender           str
age            int64
city             str
signup_date      str
dtype: object

missing values:
customer_id    0
name           0
gender         0
age            0
city           0
signup_date    0
dtype: int64

duplicated rows: 0


In [142]:
for name, df in datasets.items():
    check_data_overview(name, df)
    print()


===== customers =====
shape: (150, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
dtypes:
customer_id    int64
name             str
gender           str
age            int64
city             str
signup_date      str
dtype: object
missing:
customer_id    0
name           0
gender         0
age            0
city           0
signup_date    0
dtype: int64
duplicated rows: 0

===== products =====
shape: (100, 4)
columns: ['product_id', 'product_name', 'category', 'price']
dtypes:
product_id      int64
product_name      str
category          str
price           int64
dtype: object
missing:
product_id      0
product_name    0
category        0
price           0
dtype: int64
duplicated rows: 0

===== orders =====
shape: (300, 5)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
dtypes:
order_id                   int64
customer_id                int64
order_date        datetime64[us]
payment_method               str
order_status   

![Jupyter Notebook 데이터 구조 점검 결과 화면 예시](../book/assets/images/ch03/ch03_jupyter_data_overview_result.svg)


In [141]:
def check_data_overview(name, df, key_column=None):
    print(f"===== {name} =====")
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    print("dtypes:")
    print(df.dtypes)
    print("missing:")
    print(df.isna().sum())
    print("duplicated rows:", df.duplicated().sum())

    if key_column is not None:
        print("key missing:", df[key_column].isna().sum())
        print("key duplicated:", df[key_column].duplicated().sum())

## 17. LLM에게 데이터 구조를 설명시키는 법

LLM에게 원본 데이터를 그대로 붙여 넣는 것은 피하는 것이 좋습니다. 대신 아래처럼 구조 요약만 전달합니다.

- 파일명
- 컬럼명
- 행과 열 개수
- 데이터 타입
- 결측치 개수
- 중복 여부
- 파일 간 키 관계


### [각 데이터셋을 원본 값 없이 구조 요약만 출력하기위해 필요한 Python 코드]

In [150]:
# [각 데이터셋을 원본 값 없이 구조 요약만 출력하기위해 필요한 Python 코드]

key_columns = {
    "customers": "customer_id",
    "products": "product_id",
    "orders": "order_id",
    "order_items": "order_item_id",
}

for name, df in datasets.items():
    key = key_columns[name]

    print(f"{name}.csv 구조 요약")
    print()

    print("행/열:", df.shape)
    print("컬럼:", list(df.columns))
    print("데이터 타입:")
    print(df.dtypes.astype(str).to_dict())

    print("결측치:")
    print(df.isna().sum().to_dict())

    print("전체 중복:", df.duplicated().sum())
    print(f"{key} 중복:", df[key].duplicated().sum())
    
    print()
    print("-" * 50)
    print()


print("파일 간 키 관계")
print("customers.customer_id -> orders.customer_id")
print("orders.order_id -> order_items.order_id")
print("products.product_id -> order_items.product_id")

    

customers.csv 구조 요약

행/열: (150, 6)
컬럼: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
데이터 타입:
{'customer_id': 'int64', 'name': 'str', 'gender': 'str', 'age': 'int64', 'city': 'str', 'signup_date': 'str'}
결측치:
{'customer_id': 0, 'name': 0, 'gender': 0, 'age': 0, 'city': 0, 'signup_date': 0}
전체 중복: 0
customer_id 중복: 0

--------------------------------------------------

products.csv 구조 요약

행/열: (100, 4)
컬럼: ['product_id', 'product_name', 'category', 'price']
데이터 타입:
{'product_id': 'int64', 'product_name': 'str', 'category': 'str', 'price': 'int64'}
결측치:
{'product_id': 0, 'product_name': 0, 'category': 0, 'price': 0}
전체 중복: 0
product_id 중복: 0

--------------------------------------------------

orders.csv 구조 요약

행/열: (300, 5)
컬럼: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
데이터 타입:
{'order_id': 'int64', 'customer_id': 'int64', 'order_date': 'datetime64[us]', 'payment_method': 'str', 'order_status': 'str'}
결측치:
{'order_id': 0, 'customer_id': 

In [147]:
llm_dataset_summary = shape_summary.merge(column_summary, on="dataset")
llm_dataset_summary


,dataset,rows,columns,column_count,column_names
0,customers,150,6,6,"customer_id, name, gender, age, city, signup_date"
1,products,100,4,4,"product_id, product_name, category, price"
2,orders,300,5,5,"order_id, customer_id, order_date, payment_met..."
3,order_items,764,5,5,"order_item_id, order_id, product_id, quantity,..."


In [148]:
for _, row in llm_dataset_summary.iterrows():
    print(f"- {row['dataset']}: {row['rows']}행 {row['columns']}열")
    print(f"  컬럼: {row['column_names']}")


- customers: 150행 6열
  컬럼: customer_id, name, gender, age, city, signup_date
- products: 100행 4열
  컬럼: product_id, product_name, category, price
- orders: 300행 5열
  컬럼: order_id, customer_id, order_date, payment_method, order_status
- order_items: 764행 5열
  컬럼: order_item_id, order_id, product_id, quantity, unit_price


### 데이터 구조 설명 요청 예시

아래 프롬프트는 LLM에게 붙여 넣을 수 있는 예시입니다. 실제 데이터 전체가 아니라 구조 정보만 포함합니다.


In [149]:
prompt = f"""
온라인 쇼핑몰 데이터 분석을 시작하기 전에 다음 CSV 파일들의 구조를 이해하려고 합니다.

데이터셋 요약:
{llm_dataset_summary[['dataset', 'rows', 'columns', 'column_names']].to_string(index=False)}

파일 간 관계:
- customers.customer_id -> orders.customer_id
- orders.order_id -> order_items.order_id
- products.product_id -> order_items.product_id

요청:
1. 각 파일이 어떤 역할을 하는지 설명해 주세요.
2. 분석 전에 확인해야 할 항목을 체크리스트로 정리해 주세요.
3. 실제 데이터 확인 없이 단정한 내용과 추가 확인이 필요한 내용을 구분해 주세요.
"""

print(prompt)



온라인 쇼핑몰 데이터 분석을 시작하기 전에 다음 CSV 파일들의 구조를 이해하려고 합니다.

데이터셋 요약:
    dataset  rows  columns                                                    column_names
  customers   150        6               customer_id, name, gender, age, city, signup_date
   products   100        4                       product_id, product_name, category, price
     orders   300        5 order_id, customer_id, order_date, payment_method, order_status
order_items   764        5       order_item_id, order_id, product_id, quantity, unit_price

파일 간 관계:
- customers.customer_id -> orders.customer_id
- orders.order_id -> order_items.order_id
- products.product_id -> order_items.product_id

요청:
1. 각 파일이 어떤 역할을 하는지 설명해 주세요.
2. 분석 전에 확인해야 할 항목을 체크리스트로 정리해 주세요.
3. 실제 데이터 확인 없이 단정한 내용과 추가 확인이 필요한 내용을 구분해 주세요.



## 18. LLM 답변 검증 연습

LLM이 다음과 같이 답했다고 가정해 봅니다.

> 고객 데이터에 age 컬럼이 있으므로 연령대별 매출 분석을 바로 수행하면 됩니다.

이 답변은 그럴듯하지만 충분히 안전하지 않습니다. 아래 내용을 직접 확인해야 합니다.

- `age` 컬럼이 실제로 존재하는가?
- `age` 컬럼에 결측치나 이상치가 있는가?
- 고객 데이터와 주문 데이터가 `customer_id`로 연결되는가?
- 매출을 계산하려면 주문 상세와 상품 또는 단가 정보가 필요한가?
- 취소 주문을 포함할지 제외할지 기준이 있는가?


In [151]:
validation_check = pd.DataFrame([
    {
        "question": "customers에 age 컬럼이 있는가?",
        "result": "age" in customers.columns,
    },
    {
        "question": "age 결측치 개수는?",
        "result": customers["age"].isna().sum() if "age" in customers.columns else "컬럼 없음",
    },
    {
        "question": "orders.customer_id가 customers.customer_id와 연결되는가?",
        "result": len(invalid_customers) == 0,
    },
    {
        "question": "매출 계산에 필요한 quantity와 unit_price가 있는가?",
        "result": {"quantity", "unit_price"}.issubset(order_items.columns),
    },
])

validation_check


,question,result
0,customers에 age 컬럼이 있는가?,True
1,age 결측치 개수는?,0
2,orders.customer_id가 customers.customer_id와 연결되는가?,True
3,매출 계산에 필요한 quantity와 unit_price가 있는가?,True


## 19. 이번 장 점검 체크리스트

| 점검 항목 | 확인 |
| --- | --- |
| 필요한 CSV 파일이 모두 존재하는가? | OK |
| 각 데이터셋의 행과 열 개수를 확인했는가? | OK |
| 컬럼명이 예상과 일치하는가? | OK |
| 날짜 컬럼의 데이터 타입을 확인했는가? | OK |
| 숫자 컬럼이 실제 숫자형으로 저장되어 있는가? | OK |
| 결측치가 있는 컬럼을 확인했는가? | OK |
| 중복 데이터가 있는지 확인했는가? | OK |
| 주요 ID 컬럼의 중복 여부를 확인했는가? | OK |
| 여러 파일을 연결할 키 컬럼을 확인했는가? | OK|
| 파일 간 키 관계가 실제로 연결 가능한지 확인했는가? | OK |
| LLM에 원본 데이터 대신 구조 요약만 입력했는가? | OK|
| LLM이 제안한 설명을 실제 데이터와 비교해 검증했는가? | OK |


## 20. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 4개 CSV 파일의 행과 열 개수를 하나의 표로 정리하세요.
2. 각 파일의 결측치 개수와 결측치 비율을 확인하세요.
3. `orders.order_date`를 날짜 타입으로 변환하고 데이터 기간을 확인하세요.
4. `order_items`와 `products`를 병합해 카테고리별 주문 금액을 계산하세요.
5. LLM에게 데이터 구조 요약을 전달하는 프롬프트를 직접 작성하세요.
6. LLM이 만든 분석 아이디어가 실제 컬럼과 키 관계에 맞는지 검증하세요.

In [ ]:
# 과제 풀이 공간입니다.
# 필요한 코드를 직접 작성해 보세요.

#### 1. 4개 CSV 파일의 행과 열 개수 정리

In [152]:
shape_summary = pd.DataFrame([
    {
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
    }
    for name, df in datasets.items()
])
shape_summary

,dataset,rows,columns
0,customers,150,6
1,products,100,4
2,orders,300,5
3,order_items,764,5


### 2. 결측치 개수와 결측치 비율 확인

In [153]:
for name, df in datasets.items():
    print(f"\n===== {name} =====")
    print("결측치 개수")
    print(df.isna().sum()) #컬럼별 결측치 개수
    print("\n결측치 비율(%)")
    print((df.isna().mean() * 100).round(2)) #컬럼별 결측치 비율(%)

# 결측치 개수와 비율을 하나의 데이터프레임으로 정리
missing_summary = []
for name, df in datasets.items():
    for column in df.columns:
        missing_summary.append({
            "dataset": name,
            "column": column,
            "missing_count": df[column].isna().sum(),
            "missing_rate": round(df[column].isna().mean() * 100, 2),
        })
missing_summary = pd.DataFrame(missing_summary)
missing_summary


===== customers =====
결측치 개수
customer_id    0
name           0
gender         0
age            0
city           0
signup_date    0
dtype: int64

결측치 비율(%)
customer_id    0.0
name           0.0
gender         0.0
age            0.0
city           0.0
signup_date    0.0
dtype: float64

===== products =====
결측치 개수
product_id      0
product_name    0
category        0
price           0
dtype: int64

결측치 비율(%)
product_id      0.0
product_name    0.0
category        0.0
price           0.0
dtype: float64

===== orders =====
결측치 개수
order_id          0
customer_id       0
order_date        0
payment_method    0
order_status      0
dtype: int64

결측치 비율(%)
order_id          0.0
customer_id       0.0
order_date        0.0
payment_method    0.0
order_status      0.0
dtype: float64

===== order_items =====
결측치 개수
order_item_id    0
order_id         0
product_id       0
quantity         0
unit_price       0
dtype: int64

결측치 비율(%)
order_item_id    0.0
order_id         0.0
product_id       0.0
quant

,dataset,column,missing_count,missing_rate
0,customers,customer_id,0,0.0
1,customers,name,0,0.0
2,customers,gender,0,0.0
3,customers,age,0,0.0
4,customers,city,0,0.0
5,customers,signup_date,0,0.0
6,products,product_id,0,0.0
7,products,product_name,0,0.0
8,products,category,0,0.0
9,products,price,0,0.0


### 3. orders.order_date 날짜 변환 및 데이터 기간 확인

In [154]:
# [3orders.order_date 날짜 변환 및 데이터 기간 확인]

orders["order_date"] = pd.to_datetime(
    orders["order_date"],
    errors="coerce",
)
print("날짜 변환 실패:", orders["order_date"].isna().sum())
print("가장 빠른 주문일:", orders["order_date"].min())
print("가장 최근 주문일:", orders["order_date"].max())



날짜 변환 실패: 0
가장 빠른 주문일: 2025-09-15 00:00:00
가장 최근 주문일: 2026-09-14 00:00:00


### 4. order_items와 products 병합 후 카테고리별 주문 금액 계산

주문 상태를 고려하지 않음!!! 실제 확정 매출이 분석 대상이라면 먼저 orders와 연결하여 order_status를 반영해야 함

In [157]:
#[4. order_items와 products 병합 후 카테고리별 주문 금액 계산]

#상품정보 연결
order_items_with_products = order_items.merge(
    products,
    on="product_id",
    how="left",
)
#주문 상세별 금액 계산
order_items_with_products["line_amount"] = (
    order_items_with_products["quantity"]
    * order_items_with_products["unit_price"]
)
#카테고리별 합산
category_sales = (
    order_items_with_products
    .groupby("category", as_index=False)["line_amount"]
    .sum()
    .sort_values("line_amount", ascending=False)
)

category_sales #주문 상태를 고려하지 않음!!! 실제 확정 매출이 분석 대상이라면 먼저 orders와 연결하여 order_status를 반영해야 함


,category,line_amount
3,스포츠,50174000
1,뷰티,47551000
5,전자기기,41003000
2,생활용품,34839000
4,식품,33597000
0,도서,24645000
6,패션,23801000


#### 💡나의 필기💡 -[각 데이터셋 구조 요약 Python 코드] 

이코드 실행 결과를 붙여넣어도 됨! 🪄🪄🪄🪄

In [185]:
# [각 데이터셋 구조 요약 Python 코드]

# 주요 ID 컬럼 정의
key_columns = {
    "customers": "customer_id",
    "products": "product_id",
    "orders": "order_id",
    "order_items": "order_item_id",
}

print("===== 데이터 구조 요약 =====")

for name, df in datasets.items():
    print(f"\n{name}.csv")
    print("행/열:", df.shape)
    print("컬럼:", list(df.columns))
    print("데이터 타입:", df.dtypes.astype(str).to_dict())


print("\n===== 결측치 개수와 비율(%) =====")
for name, df in datasets.items():
    missing_table = pd.DataFrame({"결측치 개수": df.isna().sum(),"비율(%)": (df.isna().mean() * 100).round(2),})
    print(f"\n [{name}] ")
    print(missing_table)

print("\n===== 전체 행 중복 확인 =====")
for name, df in datasets.items():
    print(name, df.duplicated().sum())


# 5. 주요 ID 결측치와 중복 확인
print("\n===== 주요 ID 결측치와 중복 확인 =====")
for name, key in key_columns.items():
    df = datasets[name]
    print(name,"missing:", df[key].isna().sum(),"duplicated:", df[key].duplicated().sum(),)

print("\n===== 파일 간 키 관계 =====")
print("customers.customer_id -> orders.customer_id")
print("orders.order_id -> order_items.order_id")
print("products.product_id -> order_items.product_id")

print("\n===== 실제 키 관계 검증 =====")
invalid_customers = orders.loc[~orders["customer_id"].isin(customers["customer_id"])]
invalid_orders = order_items.loc[~order_items["order_id"].isin(orders["order_id"])]
invalid_products = order_items.loc[~order_items["product_id"].isin(products["product_id"])]
print("없는 customer_id:", len(invalid_customers))
print("없는 order_id:", len(invalid_orders))
print("없는 product_id:", len(invalid_products))

print("\n===== 날짜 타입 및 분석 기간 확인 =====")
print("변환 전 타입:", orders["order_date"].dtype)
orders["order_date"] = pd.to_datetime(orders["order_date"],errors="coerce",format="%Y-%m-%d",)
print("변환 후 타입:", orders["order_date"].dtype)
print("날짜 변환 실패:", orders["order_date"].isna().sum())
print("가장 빠른 주문일:", orders["order_date"].min())
print("가장 최근 주문일:", orders["order_date"].max())

===== 데이터 구조 요약 =====

customers.csv
행/열: (150, 6)
컬럼: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
데이터 타입: {'customer_id': 'int64', 'name': 'str', 'gender': 'str', 'age': 'int64', 'city': 'str', 'signup_date': 'str'}

products.csv
행/열: (100, 4)
컬럼: ['product_id', 'product_name', 'category', 'price']
데이터 타입: {'product_id': 'int64', 'product_name': 'str', 'category': 'str', 'price': 'int64'}

orders.csv
행/열: (300, 5)
컬럼: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
데이터 타입: {'order_id': 'int64', 'customer_id': 'int64', 'order_date': 'datetime64[us]', 'payment_method': 'str', 'order_status': 'str'}

order_items.csv
행/열: (764, 5)
컬럼: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']
데이터 타입: {'order_item_id': 'int64', 'order_id': 'int64', 'product_id': 'int64', 'quantity': 'int64', 'unit_price': 'int64'}

===== 결측치 개수와 비율(%) =====

 [customers] 
             결측치 개수  비율(%)
customer_id       0    0.0
name              0    

### 5. LLM에게 전달할 데이터 구조 요약 프롬프트 작성

온라인 쇼핑몰 데이터 분석을 시작하려고 합니다.

### 데이터셋 요약

##### customers.csv 구조 요약
- 행/열: (150, 6)
- 컬럼: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
- 데이터 타입: {'customer_id': 'int64', 'name': 'str', 'gender': 'str', 'age': 'int64', 'city': 'str', 'signup_date': 'str'}
- 결측치: {'customer_id': 0, 'name': 0, 'gender': 0, 'age': 0, 'city': 0, 'signup_date': 0}
- 전체 중복: 0
- customer_id 중복: 0

--------------------------------------------------

##### products.csv 구조 요약
- 행/열: (100, 4)
- 컬럼: ['product_id', 'product_name', 'category', 'price']
- 데이터 타입:{'product_id': 'int64', 'product_name': 'str', 'category': 'str', 'price': 'int64'}
- 결측치:{'product_id': 0, 'product_name': 0, 'category': 0, 'price': 0}
- 전체 중복: 0
- product_id 중복: 0

--------------------------------------------------

##### orders.csv 구조 요약
- 행/열: (300, 5)
- 컬럼: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
- 데이터 타입: {'order_id': 'int64', 'customer_id': 'int64', 'order_date': 'datetime64[us]', 'payment_method': 'str', 'order_status': 'str'}
- 결측치: {'order_id': 0, 'customer_id': 0, 'order_date': 0, 'payment_method': 0, 'order_status': 0}
- 전체 중복: 0
- order_id 중복: 0

--------------------------------------------------

##### order_items.csv 구조 요약
- 행/열: (764, 5)
- 컬럼: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']
- 데이터 타입:{'order_item_id': 'int64', 'order_id': 'int64', 'product_id': 'int64', 'quantity': 'int64', 'unit_price': 'int64'}
- 결측치:{'order_item_id': 0, 'order_id': 0, 'product_id': 0, 'quantity': 0, 'unit_price': 0}
- 전체 중복: 0
- order_item_id 중복: 0

--------------------------------------------------

##### 파일 간 키 관계
- customers.customer_id -> orders.customer_id
- orders.order_id -> order_items.order_id
- products.product_id -> order_items.product_id

### 참고 사항

* `order_date`는 날짜 타입으로 변환되어 있습니다.
* `signup_date`는 현재 문자열(str) 타입입니다.
* `products.price`와 `order_items.unit_price`는 서로 다른 컬럼이므로 두 값의 업무적 의미가 동일하다고 임의로 가정하지 마세요.
* `order_status`가 존재하므로 매출 분석 시 취소 또는 환불 주문의 포함 여부를 별도로 검토해야 합니다.
* 파일 간 키 관계는 위와 같이 정의되어 있지만, 모든 참조 ID가 실제로 존재하는지는 별도의 관계 검증 결과가 필요합니다.

### 요청

이 데이터 구조를 기준으로 수행 가능한 분석 아이디어를 제안해 주세요.

각 분석 아이디어마다 다음 내용을 함께 작성해 주세요.

1. 분석 목적
2. 필요한 데이터셋
3. 사용할 컬럼
4. 필요한 테이블 병합 관계
5. 계산 방법
6. 분석 전에 추가로 확인해야 할 사항
7. 현재 데이터만으로 수행 가능한지 여부

다음 조건을 반드시 지켜 주세요.

* 실제 데이터에 존재하지 않는 컬럼을 임의로 가정하지 마세요.
* 현재 제공된 정보만으로 확인할 수 없는 내용은 "추가 확인 필요"라고 표시해 주세요.
* 결측치와 중복이 없다는 사실만으로 데이터가 완전히 정상이라고 단정하지 마세요.
* `price`와 `unit_price`의 의미를 임의로 동일하다고 판단하지 마세요.
* 매출을 계산할 경우 어떤 주문 상태를 포함했는지 명확히 설명해 주세요.
* 테이블을 병합해야 하는 경우 실제 제공된 키 관계에 맞는지 확인해 주세요.


## 마무리

이번 장의 핵심은 “데이터를 불러왔다”에서 끝내지 않는 것입니다.

데이터 분석을 시작하기 전에 파일, 행과 열, 컬럼명, 데이터 타입, 결측치, 중복, 키 관계를 확인해야 이후 분석이 흔들리지 않습니다. 다음 장에서는 이 구조를 바탕으로 pandas의 선택, 필터링, 정렬, 집계 기초를 다룹니다.


#### 최종 Evidence 기록하기

[Chapter 03 Evidence]

1. CSV 파일 존재 여부

* customers.csv: PASS
* products.csv: PASS
* orders.csv: PASS
* order_items.csv: PASS

2. 데이터 규모

* customers: 150 rows × 6 cols
* products: 100 rows × 4 cols
* orders: 300 rows × 5 cols
* order_items: 764 rows × 5 cols

3. 품질 점검

* 주요 ID 결측: 0건
* 주요 ID 중복: 0건
* 전체 행 중복: 0건
* 날짜 변환 실패: 0건

4. 관계 검증
없는 customer_id: 0
없는 order_id: 0
없는 product_id: 0

5. LLM 활용

* 요청 목적: 원본 개인정보나 상세 거래값을 직접 전달하지 않고 데이터 구조와 품질 요약을 바탕으로 데이터셋의 역할, 분석 전 점검 항목, 분석 아이디어를 검토하기 위해 활용하였다.
* 실제 반영: 데이터셋의 역할과 파일 간 키 관계를 정리하고, 결측치·중복·날짜 타입·참조 관계 등 분석 전 확인 항목을 구성하는 데 활용하였다.
* 사람이 검증한 항목: 실제 파일의 행·열 개수, 컬럼명, 데이터 타입, 결측치, 전체 행 중복, 주요 ID 중복, 날짜 변환 결과, 파일 간 키 관계를 직접 코드로 확인하였다.


## 데이터 분석을 시작하기 전 꼭 해야할 데이터의 유효성 검사 - 💡나의 템플릿으로 만들자!💡

파일 존재 → 구조 → 결측치 → 중복 → PK → FK → 날짜 → 숫자값 → 최종 요약

In [ ]:
from pathlib import Path
import pandas as pd


# ============================================================
# 0. 검사 설정
# 프로젝트가 바뀌면 이 부분만 주로 수정하면 됨
# ============================================================

# CSV 파일이 들어 있는 폴더 경로
data_dir = Path("../data")

# 사용할 CSV 파일 이름 정의
file_names = {
    "customers": "customers.csv",
    "products": "products.csv",
    "orders": "orders.csv",
    "order_items": "order_items.csv",
}

# 각 데이터셋에서 하나의 행을 구분하는 주요 ID 컬럼 정의
key_columns = {
    "customers": "customer_id",
    "products": "product_id",
    "orders": "order_id",
    "order_items": "order_item_id",
}

# 데이터셋 간 연결 관계 정의
# 형식:
# (자식 테이블, 자식 키, 부모 테이블, 부모 키)
relationships = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
]

# 날짜형으로 변환해서 확인할 컬럼 정의
date_columns = {
    "customers": ["signup_date"],
    "orders": ["order_date"],
}


# ============================================================
# 1. CSV 파일 존재 여부 확인
# ============================================================

print("===== 1. CSV 파일 존재 여부 =====")

# 파일별 존재 여부를 저장할 딕셔너리
file_status = {}

# 파일을 하나씩 확인
for name, file_name in file_names.items():

    # 데이터 폴더와 파일명을 합쳐 실제 경로 생성
    file_path = data_dir / file_name

    # 파일이 실제로 존재하는지 확인
    exists = file_path.exists()

    # 존재 여부 저장
    file_status[name] = exists

    # 존재하면 PASS, 없으면 FAIL 출력
    print(
        f"{file_name}:",
        "PASS" if exists else "FAIL"
    )


# ============================================================
# 2. CSV 파일 불러오기
# ============================================================

# 불러온 데이터프레임을 저장할 딕셔너리
datasets = {}

for name, file_name in file_names.items():

    # 실제 CSV 파일 경로 생성
    file_path = data_dir / file_name

    # 파일이 존재하는 경우에만 읽기
    if file_path.exists():

        # CSV를 읽어서 datasets 딕셔너리에 저장
        datasets[name] = pd.read_csv(file_path)


# 자주 사용할 데이터프레임을 변수로 꺼냄
customers = datasets["customers"]
products = datasets["products"]
orders = datasets["orders"]
order_items = datasets["order_items"]


# ============================================================
# 3. 데이터 구조 확인
# 행/열 개수, 컬럼명, 데이터 타입 확인
# ============================================================

print("\n===== 2. 데이터 구조 =====")

for name, df in datasets.items():

    # 현재 확인 중인 데이터셋 이름 출력
    print(f"\n[{name}]")

    # 행과 열 개수 확인
    print("행/열:", df.shape)

    # 컬럼명 확인
    print("컬럼:", list(df.columns))

    # 각 컬럼의 데이터 타입 확인
    print(
        "데이터 타입:",
        df.dtypes.astype(str).to_dict()
    )


# ============================================================
# 4. 결측치 개수와 비율 확인
# ============================================================

print("\n===== 3. 결측치 확인 =====")

for name, df in datasets.items():

    # 결측치 개수와 비율을 하나의 표로 생성
    missing_table = pd.DataFrame({

        # 각 컬럼의 결측치 개수
        "missing_count":
            df.isna().sum(),

        # 각 컬럼의 결측치 비율(%)
        "missing_rate(%)":
            (df.isna().mean() * 100).round(2),
    })

    # 데이터셋 이름 출력
    print(f"\n[{name}]")

    # 결측치 요약표 출력
    print(missing_table)


# ============================================================
# 5. 전체 행 중복 확인
# ============================================================

print("\n===== 4. 전체 행 중복 =====")

for name, df in datasets.items():

    # 행 전체가 완전히 동일한 중복 행의 개수 계산
    duplicated_count = df.duplicated().sum()

    # 결과 출력
    print(
        name,
        "duplicated rows:",
        duplicated_count
    )


# ============================================================
# 6. 주요 ID 결측치와 중복 확인
# ============================================================

print("\n===== 5. 주요 ID 검사 =====")

for name, key in key_columns.items():

    # 해당 데이터셋 가져오기
    df = datasets[name]

    # 주요 ID 컬럼의 결측치 개수 확인
    missing_count = df[key].isna().sum()

    # 주요 ID 컬럼의 중복 개수 확인
    duplicated_count = df[key].duplicated().sum()

    # 결과 출력
    print(
        f"{name}.{key}",
        "| missing:", missing_count,
        "| duplicated:", duplicated_count,
    )


# ============================================================
# 7. 파일 간 키 관계 검증
# 자식 테이블의 ID가 부모 테이블에 실제 존재하는지 확인
# ============================================================

print("\n===== 6. 파일 간 관계 검증 =====")

for (
    child_table,
    child_key,
    parent_table,
    parent_key
) in relationships:

    # 자식 데이터프레임 가져오기
    child_df = datasets[child_table]

    # 부모 데이터프레임 가져오기
    parent_df = datasets[parent_table]

    # 자식 키 값이 부모 키 목록에 없는 행만 추출
    invalid_rows = child_df.loc[
        ~child_df[child_key].isin(
            parent_df[parent_key]
        )
    ]

    # 검사한 관계 출력
    print(
        f"{child_table}.{child_key}"
        f" -> "
        f"{parent_table}.{parent_key}"
    )

    # 부모 테이블에 존재하지 않는 참조값 개수 출력
    print(
        "존재하지 않는 참조값:",
        len(invalid_rows)
    )


# ============================================================
# 8. 날짜 타입 변환과 변환 실패 확인
# ============================================================

print("\n===== 7. 날짜 검사 =====")

for name, columns in date_columns.items():

    # 현재 데이터셋 가져오기
    df = datasets[name]

    # 날짜 컬럼을 하나씩 검사
    for column in columns:

        print(f"\n[{name}.{column}]")

        # 변환 전 데이터 타입 확인
        print(
            "변환 전 타입:",
            df[column].dtype
        )

        # 날짜형으로 변환
        # 변환할 수 없는 값은 NaT로 처리
        converted = pd.to_datetime(
            df[column],
            errors="coerce"
        )

        # 원래 값은 있었지만 날짜 변환 후 NaT가 된 경우만
        # 실제 날짜 변환 실패로 판단
        conversion_failed = (
            df[column].notna()
            & converted.isna()
        )

        # 날짜 변환 실패 개수 출력
        print(
            "날짜 변환 실패:",
            conversion_failed.sum()
        )

        # 가장 이른 날짜 확인
        print(
            "가장 빠른 날짜:",
            converted.min()
        )

        # 가장 최근 날짜 확인
        print(
            "가장 최근 날짜:",
            converted.max()
        )

        # 검증 후 실제 데이터프레임의 컬럼도 날짜형으로 변경
        df[column] = converted

        # 변환 후 데이터 타입 확인
        print(
            "변환 후 타입:",
            df[column].dtype
        )


# ============================================================
# 9. 숫자형 데이터 기본 유효성 검사
# 업무 규칙에 따라 조건은 변경 가능
# ============================================================

print("\n===== 8. 숫자값 유효성 검사 =====")

# 나이가 0 이하인 데이터 개수 확인
print(
    "age <= 0:",
    (customers["age"] <= 0).sum()
)

# 상품 가격이 음수인 데이터 개수 확인
print(
    "price < 0:",
    (products["price"] < 0).sum()
)

# 주문 수량이 0 이하인 데이터 개수 확인
print(
    "quantity <= 0:",
    (order_items["quantity"] <= 0).sum()
)

# 주문 단가가 음수인 데이터 개수 확인
print(
    "unit_price < 0:",
    (order_items["unit_price"] < 0).sum()
)


# ============================================================
# 10. 범주형 데이터 값 확인
# 오타, 표기 차이, 예상하지 못한 값 확인
# ============================================================

print("\n===== 9. 범주형 값 확인 =====")

# 확인할 범주형 컬럼 정의
categorical_columns = {
    "customers": [
        "gender",
        "city",
    ],
    "products": [
        "category",
    ],
    "orders": [
        "payment_method",
        "order_status",
    ],
}

for name, columns in categorical_columns.items():

    # 현재 데이터셋 가져오기
    df = datasets[name]

    for column in columns:

        # 현재 검사하는 컬럼 표시
        print(f"\n[{name}.{column}]")

        # 각 값이 몇 번 등장하는지 확인
        # dropna=False로 결측치도 함께 확인
        print(
            df[column].value_counts(
                dropna=False
            )
        )


# ============================================================
# 11. Evidence 제출용 최종 요약
# 앞에서 검사한 결과를 짧게 정리
# ============================================================

print("\n" + "=" * 60)
print("데이터 유효성 검사 최종 요약")
print("=" * 60)


# ============================================================
# 11-1. CSV 파일 존재 여부 요약
# ============================================================

print("\n1. CSV 파일 존재 여부")

for name, file_name in file_names.items():

    print(
        f"- {file_name}:",
        "PASS"
        if file_status[name]
        else "FAIL"
    )


# ============================================================
# 11-2. 데이터 규모 요약
# ============================================================

print("\n2. 데이터 규모")

for name, df in datasets.items():

    # 행 수 × 열 수 형태로 출력
    print(
        f"- {name}: "
        f"{df.shape[0]} rows × "
        f"{df.shape[1]} cols"
    )


# ============================================================
# 11-3. 주요 품질 점검 합계
# ============================================================

# 모든 주요 ID 컬럼의 결측치 개수를 합산
id_missing_total = sum(
    datasets[name][key]
    .isna()
    .sum()

    for name, key
    in key_columns.items()
)

# 모든 주요 ID 컬럼의 중복 개수를 합산
id_duplicate_total = sum(
    datasets[name][key]
    .duplicated()
    .sum()

    for name, key
    in key_columns.items()
)

# 모든 데이터셋의 전체 중복 행 개수를 합산
row_duplicate_total = sum(
    df.duplicated().sum()

    for df
    in datasets.values()
)


print("\n3. 품질 점검")

print(
    "- 주요 ID 결측:",
    f"{id_missing_total}건"
)

print(
    "- 주요 ID 중복:",
    f"{id_duplicate_total}건"
)

print(
    "- 전체 행 중복:",
    f"{row_duplicate_total}건"
)


# ============================================================
# 11-4. 파일 간 관계 검증 요약
# ============================================================

print("\n4. 관계 검증")

for (
    child_table,
    child_key,
    parent_table,
    parent_key
) in relationships:

    # 자식 키가 부모 키 목록에 없는 개수 계산
    invalid_count = (
        ~datasets[child_table][child_key]
        .isin(
            datasets[parent_table][parent_key]
        )
    ).sum()

    # 결과 출력
    print(
        f"- 없는 {child_key}:",
        invalid_count
    )